In [4]:
%%writefile add_integers.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void add_arrays(int *c, const int *a, const int *b, int size)
{
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < size) {
    c[i] = a[i] + b[i];
  }
}

int main() {
  const int size = 5;
  int a[size] = {1, 2, 3, 4, 5};
  int b[size] = {1, 2, 3, 4, 5};

  int *d_a, *d_b, *d_c;

  // Allocate GPU memory using correct size (size * sizeof(int))
  cudaMalloc((void**)&d_c, size * sizeof(int));
  cudaMalloc((void**)&d_a, size * sizeof(int));
  cudaMalloc((void**)&d_b, size * sizeof(int));

  // Copy data from Host to Device
  cudaMemcpy(d_a, a, size * sizeof(int), cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, b, size * sizeof(int), cudaMemcpyHostToDevice);

  // Launch kernel (2 blocks, 3 threads per block = 6 threads for 5 elements)
  add_arrays<<<2, 3>>>(d_c, d_a, d_b, size);

  // Synchronize CPU and GPU
  cudaDeviceSynchronize();

  // Copy the result back to Host
  int *c = (int*) malloc(size * sizeof(int));
  cudaMemcpy(c, d_c, size * sizeof(int), cudaMemcpyDeviceToHost);

  // Print the result
  printf("Result: ");
  for (int i = 0; i < size; i++) {
    printf("%d ", c[i]);
  }
  printf("\n");

  // Free Host and Device Memory
  free(c);       // Use standard free() for host memory
  cudaFree(d_a); // Use cudaFree() for device memory
  cudaFree(d_b);
  cudaFree(d_c);

  return 0;
}

Writing add_integers.cu


In [5]:
!nvcc add_integers.cu -o add_integers

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
!./add_integers

Result: 2 4 6 8 10 
